# Part 1 — Notebook 01: MadGraph Process Setup and LHE Generation

## Pedagogical Goal & Overview

Welcome to **Part 1 — Notebook 01** of the experimental High-Energy Physics (HEP) Monte Carlo training series.

In this notebook, you will execute the first hands-on step of event generation:
$$\text{Process Definition} \rightarrow \text{MadGraph Process/Run Cards} \rightarrow \text{Parton-Level Events (LHE)} \rightarrow \text{Event Record Inspection}$$

### Primary Pedagogical Process
We study proton-proton production of a $Z$ boson recoiling against a hard matrix-element jet, with the $Z$ boson forced to decay to a bottom-quark pair:
```text
p p > z j, z > b b~
```

### Full Monte Carlo Event Pipeline Context
It is critical to understand where this notebook fits in the experimental physics chain:
$$\underbrace{\text{Hard Scattering (ME)} \rightarrow \text{Resonance Decay}}_{\text{Notebook 01 (MadGraph LHE)}} \rightarrow \text{Parton Shower} \rightarrow \text{Hadronization} \rightarrow \text{Detector Sim} \rightarrow \text{Reconstruction} \rightarrow \text{Analysis}$$

In this notebook, we operate purely at the **parton-level matrix element (ME)** stage.

## Step 1: Integrated Physics Foundations & Standard Model Review

Welcome! Before launching event generators or writing physics configuration cards, let's establish the fundamental particle physics framework.

### 1.1 What is the Standard Model (SM)?
The **Standard Model of Particle Physics** is the theoretical framework describing all known fundamental particles and three of the four fundamental forces governing our Universe: electromagnetism, the weak nuclear force, and the strong nuclear force (gravity is described separately by General Relativity).

Everything in the Universe is built from **Fermions** (matter particles with half-integer spin $s = 1/2$), which interact with one another by exchanging **Bosons** (force carriers with integer spin):

1. **Matter Particles (Fermions — Spin 1/2)**:
   - **Quarks (6 Flavors)**: Up ($u$), Down ($d$), Charm ($c$), Strange ($s$), Top ($t$), and Bottom ($b$). Quarks feel the **strong nuclear force** (mediated by gluons) and carry color charge. Quarks never exist in isolation; they are bound together inside composite hadrons (like protons and neutrons).
   - **Leptons (6 Flavors)**: Charged leptons (electron $e^-$, muon $\mu^-$, tau $\tau^-$) and their neutral partners (electron neutrino $\nu_e$, muon neutrino $\nu_\mu$, tau neutrino $\nu_\tau$). Leptons do *not* feel the strong force.
2. **Force Carriers (Gauge Bosons — Spin 1)**:
   - **Photon ($\gamma$)**: Mediates the **electromagnetic force** acting on electrically charged particles.
   - **Gluons ($g$, 8 types)**: Mediate the **strong force** holding quarks together inside protons and neutrons.
   - **$W^\pm$ and $Z^0$ Bosons**: Mediate the **weak nuclear force** responsible for radioactive beta decay and heavy resonance decays. Unlike photons and gluons, $W$ and $Z$ bosons are very massive ($m_W \approx 80.38\text{ GeV}/c^2$, $m_Z \approx 91.19\text{ GeV}/c^2$).
3. **Higgs Boson ($H^0$ — Spin 0)**:
   - Discovered at CERN's Large Hadron Collider (LHC) in 2012, the scalar Higgs boson gives mass to elementary particles via the Brout-Englert-Higgs mechanism.

![The Standard Model of Elementary Particles](standard_model_chart.svg)

---

### 1.2 Monte Carlo Particle Identifiers: The PDG ID Scheme
When computer simulation programs (like MadGraph, Pythia, or GEANT4) track particles, using text strings like `"bottom anti-quark"` is inefficient. Instead, high-energy physics uses a universal standardized numbering scheme defined by the **Particle Data Group (PDG)**.

Each elementary particle has an assigned integer **PDG ID (PID)**. Anti-particles carry a negative sign (e.g. $b = +5$, $\bar{b} = -5$).

| Particle Category | Particle Name | Symbol | PDG ID (PID) | Electric Charge ($e$) | Rest Mass ($m$) |
| :--- | :--- | :---: | :---: | :---: | :---: |
| **Quarks** | Down / Up | $d$ / $u$ | $+1$ / $+2$ | $-1/3$ / $+2/3$ | $\approx 4.67\text{ MeV}/c^2$ / $\approx 2.16\text{ MeV}/c^2$ |
| | Strange / Charm | $s$ / $c$ | $+3$ / $+4$ | $-1/3$ / $+2/3$ | $\approx 93.4\text{ MeV}/c^2$ / $\approx 1.27\text{ GeV}/c^2$ |
| | Bottom / Top | $b$ / $t$ | $+5$ / $+6$ | $-1/3$ / $+2/3$ | $\approx 4.18\text{ GeV}/c^2$ / $\approx 172.69\text{ GeV}/c^2$ |
| **Leptons** | Electron / $e$-Neutrino | $e^-$ / $\nu_e$ | $+11$ / $+12$ | $-1$ / $0$ | $0.511\text{ MeV}/c^2$ / $\approx 0\text{ (sub-eV scale)}$ |
| | Muon / $\mu$-Neutrino | $\mu^-$ / $\nu_\mu$ | $+13$ / $+14$ | $-1$ / $0$ | $105.66\text{ MeV}/c^2$ / $\approx 0\text{ (sub-eV scale)}$ |
| | Tau / $\tau$-Neutrino | $\tau^-$ / $\nu_\tau$ | $+15$ / $+16$ | $-1$ / $0$ | $1.777\text{ GeV}/c^2$ / $\approx 0\text{ (sub-eV scale)}$ |
| **Gauge Bosons** | Gluon | $g$ | $21$ | $0$ | $0\text{ (massless)}$ |
| | Photon | $\gamma$ | $22$ | $0$ | $0\text{ (massless)}$ |
| | $Z$ Boson | $Z^0$ | $23$ | $0$ | $m_Z = 91.1876\text{ GeV}/c^2$ |
| | $W^+$ / $W^-$ Boson | $W^+$ / $W^-$ | $+24$ / $-24$ | $+1$ / $-1$ | $m_W = 80.377\text{ GeV}/c^2$ |
| **Higgs Boson** | Higgs Boson | $H^0$ | $25$ | $0$ | $m_H = 125.25\text{ GeV}/c^2$ |

*Note: Anti-particles carry negated PID numbers (e.g. anti-up $\bar{u} = -2$, anti-bottom $\bar{b} = -5$, positron $e^+ = -11$). Neutrinos are treated as massless in the Standard Model ($m_\nu \approx 0$), with tiny experimental mass upper bounds below $1\text{ eV}/c^2$.*

---

### 1.3 Vector Boson Production ($pp \to Z + j$)
In proton-proton collisions at CERN's Large Hadron Collider ($\sqrt{s} = 13\text{ TeV}$), protons are composite bound states of valence quarks ($uud$), sea quarks ($q\bar{q}$ pairs), and gluons ($g$).

At leading order (LO) in Quantum Chromodynamics (QCD), a $Z$ boson is produced together with a recoiling hard matrix-element parton ($j \in \{u, d, c, s, g\}$) through three primary parton-level subprocesses:
$$q + \bar{q} \rightarrow Z + g, \quad q + g \rightarrow Z + q, \quad \bar{q} + g \rightarrow Z + \bar{q}$$

---

### 1.4 Boosted $Z \to b\bar{b}$ Decays & Angular Separation
The $Z$ boson decays to bottom-antibottom quark pairs ($Z \to b\bar{b}$) with a branching ratio of $\text{BR}(Z \to b\bar{b}) \approx 15.1\%$.

When the recoiling matrix-element parton imparts a high transverse momentum boost $p_T^Z$ to the $Z$ boson, relativistic kinematics causes the daughter bottom quarks ($b$ and $\bar{b}$) to be collimated into a tight angular opening cone $\Delta R_{b\bar{b}}$:
$$\Delta R_{b\bar{b}} \approx \frac{2 m_Z}{p_T^Z}$$

Where $\Delta R = \sqrt{(\Delta\eta)^2 + (\Delta\phi)^2}$ measures angular distance in pseudorapidity ($\eta$) and azimuthal angle ($\phi$).

- **Resolved Topology (Low $p_T^Z \ll 100\text{ GeV}$)**: Decay products fly apart with wide opening angles ($\Delta R_{b\bar{b}} > 1.5$), forming two well-separated partons.
- **Boosted Topology (High $p_T^Z \ge 150\text{ GeV}$)**: High relativistic boost collimates the decay products into a tight two-prong cone ($\Delta R_{b\bar{b}} \le 0.8$).

---

### 1.5 Generator Level vs. Reconstruction Pipeline
It is critical to understand where this notebook fits in the full experimental Monte Carlo pipeline:
$$\underbrace{\text{Hard Scattering (ME)} \rightarrow \text{Resonance Decay}}_{\text{Notebook 01 (MadGraph LHE)}} \rightarrow \text{Parton Shower} \rightarrow \text{Hadronization} \rightarrow \text{Detector Sim} \rightarrow \text{Reconstruction} \rightarrow \text{Analysis}$$

In **Part 1**, we operate purely at the **parton-level matrix element (ME)** stage, inspecting the raw Les Houches Event (LHE) generator truth record before hadronization or detector resolution effects occur.




## Step 2: Environment Setup & Workspace Initialization

Choose your execution environment using the `USE_COLAB` flag in the cell below:
- **Google Colab Mode (`USE_COLAB = True`)**: Mounts Google Drive at `/content/drive` so all MadGraph installations, process cards, and generated LHE event files persist across session restarts (`/content/drive/MyDrive/MadGraph_Zbb_Outputs`).
- **Local Machine Mode (`USE_COLAB = False`)**: Set `LOCAL_OUTPUT_DIR` to a base directory path (or leave as `None` to use current directory). Output files will be stored in `LOCAL_OUTPUT_DIR/MadGraph_Zbb_Outputs` — no Google Drive or Colab dependencies required.


In [ ]:
import os
import sys
import shutil
import subprocess

# ── Execution Environment Mode Choice ──────────────────────────────────────────
# Set USE_COLAB = True when running on Google Colab to mount Google Drive.
# Set USE_COLAB = False when running locally on your own computer/laptop.
USE_COLAB = True  # <── Change to False if running locally on your machine

# For Local Mode (USE_COLAB = False), set LOCAL_OUTPUT_DIR to a base directory path,
# or leave as None to use the current working directory.
# Output files will be stored in: LOCAL_OUTPUT_DIR + "/MadGraph_Zbb_Outputs"
LOCAL_OUTPUT_DIR = None  # <── e.g., "/path/to/your/base/dir" or None

print("=== Environment Verification & Setup ===")
print(f"Execution Mode : {'Google Colab' if USE_COLAB else 'Local Machine'}")
print(f"Python version : {sys.version.split()[0]}")

# ── 1. Configure Output & MadGraph Directories based on Mode ─────────────────
if USE_COLAB:
    if os.path.exists("/content"):
        try:
            from google.colab import drive
            print("\nMounting Google Drive for persistent storage...")
            drive.mount('/content/drive', force_remount=False)
            output_dir = "/content/drive/MyDrive/MadGraph_Zbb_Outputs"
        except Exception as e:
            print(f"Notice: Drive mount skipped ({e}). Using local workspace directory.")
            output_dir = os.path.join(os.getcwd(), "MadGraph_Zbb_Outputs")
    else:
        output_dir = os.path.join(os.getcwd(), "MadGraph_Zbb_Outputs")
else:
    base_dir = os.path.abspath(os.path.expanduser(LOCAL_OUTPUT_DIR)) if LOCAL_OUTPUT_DIR else os.getcwd()
    output_dir = os.path.join(base_dir, "MadGraph_Zbb_Outputs")

os.makedirs(output_dir, exist_ok=True)
os.chdir(output_dir)

mg5_dir = os.path.join(output_dir, "MG5_aMC_v3_5_16")
if not os.path.exists(mg5_dir) and os.path.exists(os.path.join(output_dir, "MG5_aMC")):
    mg5_dir = os.path.join(output_dir, "MG5_aMC")

print(f"Persistent Working Directory: {os.getcwd()}")
print(f"MadGraph Installation Target: {mg5_dir}")

# ── 2. Verify System Dependencies (gfortran compiler) ────────────────────────
if shutil.which("gfortran"):
    try:
        gfort_ver = subprocess.check_output(["gfortran", "--version"]).decode('utf-8').split('\n')[0]
        print(f"GFortran Compiler Found      : {gfort_ver}")
    except Exception:
        print("GFortran Compiler Found      : OK")
else:
    if USE_COLAB:
        print("Installing gfortran compiler via apt-get (~5s)...")
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "gfortran"], check=False)
    else:
        print("WARNING: 'gfortran' compiler not found on your system PATH!")
        print("MadGraph requires gfortran to compile matrix element C/Fortran code.")
        print("Please install gfortran via your OS package manager (e.g. 'sudo apt install gfortran' or 'brew install gcc').")


## Step 3: Download and Install MadGraph5_aMC@NLO

We download the official MadGraph5_aMC@NLO release archive (`MG5_aMC_v3.5.16.tar.gz`), extract it directly into your persistent output workspace directory (`MG5_aMC_v3_5_16`), and confirm that the executable `./bin/mg5_aMC` exists.


In [ ]:
mg5_tar = os.path.join(output_dir, "MG5_aMC_v3.5.16.tar.gz")
mg5_url = "https://launchpad.net/mg5amcnlo/3.0/3.7.x/+download/MG5_aMC_v3.5.16.tar.gz"

mg5_dir = os.path.join(output_dir, "MG5_aMC_v3_5_16")
if not os.path.exists(mg5_dir) and os.path.exists(os.path.join(output_dir, "MG5_aMC")):
    mg5_dir = os.path.join(output_dir, "MG5_aMC")

if not os.path.exists(mg5_dir):
    print(f"Downloading MadGraph5_aMC@NLO v3.5.16 to persistent workspace ({output_dir})...")
    !wget -q {mg5_url} -O {mg5_tar}
    print("Extracting archive into persistent workspace...")
    !tar -xzf {mg5_tar} -C {output_dir}
    print(f"MadGraph5 extracted to {mg5_dir}")
else:
    print(f"MadGraph5 already installed in persistent workspace at {mg5_dir}")

mg5_exe = os.path.join(mg5_dir, "bin", "mg5_aMC")
assert os.path.exists(mg5_exe), f"ERROR: MadGraph executable not found at {mg5_exe}"
print(f"SUCCESS: Verified MadGraph executable at {mg5_exe}")


## Step 4: Create & Review MadGraph Cards

MadGraph uses two plain-text configuration files ("cards") to fully specify event generation.
In Colab we create them directly from Python so you can inspect and modify every parameter.

- **Process Card** (`cards/zbbj_proc_card.dat`): Defines *what* to generate — the physics process, decay chain, and output directory name.
- **Run Card** (`cards/zbbj_run_card.dat`): Defines *how* to generate — beam energies, event count, random seed, and all kinematic selection cuts.

Run the cells below to write the cards to disk, then read and inspect their contents.

In [ ]:
import os

# ── Create the cards directory inside output_dir ───────────────────────────
cards_dir = os.path.join(output_dir, "cards")
os.makedirs(cards_dir, exist_ok=True)

# ── Process Card ──────────────────────────────────────────────────────────────
# Each line is a MadGraph5 command executed in sequence:
#   import model sm   : load the Standard Model particle content & Feynman rules
#   generate ...      : specify the hard-scattering process
#                       "p p > z j"  — a Z boson produced with a hard jet
#                       ", z > b b~" — forced decay of the Z to a b-bbar pair
#   output Zbbj_LO    : write the compiled matrix-element code to this directory
#   launch Zbbj_LO    : run the generation using the settings below

proc_card_lines = [
    "import model sm\n",
    "generate p p > z j, z > b b~\n",
    "output Zbbj_LO\n",
]
proc_card_content = ''.join(proc_card_lines)

proc_card_path = os.path.join(cards_dir, "zbbj_proc_card.dat")
with open(proc_card_path, "w") as f:
    f.write(proc_card_content)

# ──── Print the card so you can inspect (and edit the list above if needed) ───
print(f"=== Process Card written to: {proc_card_path} ===\n")
print(proc_card_content)


In [ ]:
# ── Official MadGraph5_aMC@NLO Run Card ──────────────────────────────────────
# Controls numerical parameters, collision energy, PDFs, and kinematic cuts.

nevents = -1     # -1 = controlled at launch time via 'set nevents N'
iseed   = 0      # Random seed (0 = auto-generate)
ptj_cut = 150.0  # Minimum pT of the recoiling jet [GeV]
ptZ_cut = 150.0  # Minimum pT of the Z boson (PDG 23) [GeV]

# ── Build the official MadGraph run_card.dat content ─────────────────────────
run_card_lines = [
    "#*********************************************************************\n",
    "#                       MadGraph5_aMC@NLO                            *\n",
    "#                     run_card.dat MadEvent                          *\n",
    "# Process: p p > z j, z > b b~ (LO)                                  *\n",
    "#*********************************************************************\n",
    "# Number of events and rnd seed                                      *\n",
    "#*********************************************************************\n",
    f"  {nevents:<10} = nevents ! Number of unweighted events requested\n",
    f"  {iseed:<10} = iseed ! rnd seed (0=assigned automatically=default))\n",
    "#*********************************************************************\n",
    "# Collider type and energy                                           *\n",
    "#*********************************************************************\n",
    "  1	= lpp1 ! beam 1 type (1 = proton)\n",
    "  1	= lpp2 ! beam 2 type (1 = proton)\n",
    "  6500.0	= ebeam1 ! beam 1 total energy in GeV (13 TeV CoM)\n",
    "  6500.0	= ebeam2 ! beam 2 total energy in GeV\n",
    "#*********************************************************************\n",
    "# PDF CHOICE                                                         *\n",
    "#*********************************************************************\n",
    "  nn23lo1	= pdlabel1 ! PDF type for beam #1\n",
    "  nn23lo1	= pdlabel2 ! PDF type for beam #2\n",
    "  230000	= lhaid ! LHAPDF ID\n",
    "#*********************************************************************\n",
    "# Renormalization and factorization scales                           *\n",
    "#*********************************************************************\n",
    "  False	= fixed_ren_scale\n",
    "  False	= fixed_fac_scale1\n",
    "  False	= fixed_fac_scale2\n",
    "  -1	= dynamical_scale_choice ! -1 = default central scale\n",
    "  1.0	= scalefact\n",
    "#*********************************************************************\n",
    "# Standard Cuts & Boosted Selection                                  *\n",
    "#*********************************************************************\n",
    f"  {ptj_cut}	= ptj ! minimum pt for the jets\n",
    "  0.0	= ptb ! minimum pt for the b\n",
    "  -1.0	= ptjmax ! maximum pt for the jets\n",
    "  -1.0	= ptbmax ! maximum pt for the b\n",
    f"  {{23: {ptZ_cut}}}	= pt_min_pdg ! pt cut for Z boson (PDG 23)\n",
    "  {}	= pt_max_pdg\n",
    "#*********************************************************************\n",
    "# Maximum and minimum absolute rapidity                              *\n",
    "#*********************************************************************\n",
    "  5.0	= etaj ! max rap for the jets\n",
    "  5.0	= etab ! max rap for the b\n",
    "  0.0	= etabmin ! min rap for the b\n",
    "#*********************************************************************\n",
    "# Minimum and maximum DeltaR distance & invariant mass for b-pairs   *\n",
    "#*********************************************************************\n",
    "  0.0	= drbb ! min distance between b's\n",
    "  0.0	= drbj ! min distance between b and jet\n",
    "  0.0	= mmbb ! min invariant mass of a b pair\n",
    "  -1.0	= mmbbmax ! max invariant mass of a b pair\n",
    "  0.0	= xptb ! minimum pt for at least one b\n",
    "#*********************************************************************\n",
    "# Maximum pdg code for quark to be considered as a light jet         *\n",
    "#*********************************************************************\n",
    "  4	= maxjetflavor ! 4 = u,d,s,c are light jets, b is heavy\n",
]
run_card_content = ''.join(run_card_lines)

run_card_path = os.path.join(cards_dir, "zbbj_run_card.dat")
with open(run_card_path, "w") as f:
    f.write(run_card_content)

print(f"=== Official Run Card written to: {run_card_path} ===\n")
print(run_card_content)


---

> [!IMPORTANT]
> ### Exercise 4: Card Parameter Review & Understanding
> Inspect the card contents printed above and answer the following questions:
> 1. What collision center-of-mass energy ($\sqrt{s}$) is configured by `ebeam1 = 6500` and `ebeam2 = 6500`?
> 2. What does the comma in `generate p p > z j, z > b b~` do?
> 3. How do `ptj = 150` and `{23: 150} = pt_min_pdg` enforce a boosted $Z$ boson ($p_T^Z \ge 150\text{ GeV}$)?

<details>
<summary>Click to show Exercise 4 Reference Solution</summary>

<p>1. <b>Collision CoM Energy</b>: $\sqrt{s} = 6500\text{ GeV} + 6500\text{ GeV} = 13000\text{ GeV} = 13\text{ TeV}$.</p>
<p>2. <b>Decay Syntax</b>: The comma <code>, z &gt; b b~</code> instructs MadGraph to force the intermediate $Z$ boson to decay into $b\bar{b}$, rather than letting it remain stable.</p>
<p>3. <b>Boost Cut Purpose</b>: <code>ptj = 150.0</code> sets the minimum $p_T$ for the recoiling light jet, while <code>{23: 150.0} = pt_min_pdg</code> explicitly sets the minimum $p_T$ for the $Z$ boson (PDG 23). Together with 2-to-2 momentum conservation in $p p \to Z j$, this guarantees a boosted $Z$ topology where both $b$ quarks are collimated.</p>

</details>

## Step 5: Test Run — 10 Events & Execution Log Reading

Before running the full production, we do a fast **10-event test** to confirm the setup compiles and runs correctly.
We reuse the cards written in Step 4 and override `nevents` to `10` at launch time — no need to rewrite the card.

### What to look for in the log
1. Feynman diagram generation and matrix-element compilation.
2. Phase space integration and cross-section ($\sigma$ in pb).
3. Event generation and compression into `.lhe.gz`.

In [ ]:
# ── Test run: 10 events ───────────────────────────────────────────────────────
# Executes the process card from Step 4 and passes launch-time overrides.
import os
import shutil

test_nevents = 10   # <── change this if you want more/fewer test events

zbbj_test_dir = os.path.join(output_dir, "Zbbj_test")

# 1. Read process card created in Step 4 and redirect output to Zbbj_test
with open(proc_card_path) as f:
    proc_content = f.read()

test_proc_card_path = os.path.join(output_dir, "cards/zbbj_test_proc_card.dat")
with open(test_proc_card_path, "w") as f:
    f.write(proc_content.replace("output Zbbj_LO", f"output {zbbj_test_dir}").replace("launch Zbbj_LO", ""))

# 2. Execute process card with MadGraph to compile matrix elements
print(f"Executing process card ({test_proc_card_path})...")
!{mg5_exe} {test_proc_card_path}

# 3. Copy custom run card from Step 4 into Zbbj_test/Cards/run_card.dat so ALL parameters take effect
target_test_run_card = os.path.join(zbbj_test_dir, "Cards", "run_card.dat")
shutil.copyfile(run_card_path, target_test_run_card)

# 4. Create batch launch script for MadGraph
batch_test_cmd = (
    f"launch {zbbj_test_dir}\n"
    f"set nevents {test_nevents}\n"
    f"set iseed 0\n"
    f"done\n"
)
batch_test_file = os.path.join(output_dir, "cards/launch_test.mg5")
with open(batch_test_file, "w") as f:
    f.write(batch_test_cmd)

# 5. Launch test event generation using MadGraph batch script
print(f"Executing {test_nevents}-event test run...")
!{mg5_exe} {batch_test_file}

test_lhe = os.path.join(zbbj_test_dir, "Events/run_01/unweighted_events.lhe.gz")
assert os.path.exists(test_lhe), f"ERROR: Test LHE file missing at {test_lhe}"
print(f"\nSUCCESS: Test run completed. LHE file: {test_lhe}")


> [!IMPORTANT]
> ### Self-Reflection, Log & Cards Checkpoint 5.1
> 1. Read the stdout execution log above. What cross section ($\sigma$) was calculated by MadGraph for $pp \to Z+j, Z \to b\bar{b}$ with $p_T^j > 150\text{ GeV}$?
> 2. How many Feynman diagrams were generated for this hard scattering process?
> 3. **Output Cards Inspection**: Check the process `Cards/` directory (`Zbbj_test/Cards/` inside your output folder). What cards and configuration files (`run_card.dat`, `proc_card.dat`, `param_card.dat`, `me5_configuration.txt`) were generated by MadGraph? Verify that your custom `run_card.dat` parameter settings (e.g. `ptj = 150.0`) are present inside `Zbbj_test/Cards/run_card.dat`.

<details>
<summary>Click to show Log Checkpoint 5.1 Reference Solution</summary>

<p>1. <b>Calculated Cross Section</b>: Found in the launch phase space integration step in the log: <code>Cross-section : X.XX +- Y.YY (pb)</code>.</p>
<p>2. <b>Feynman Diagrams</b>: Output near the top of process generation: <code>X diagrams generated</code>.</p>
<p>3. <b>Process Cards Directory</b>: Open <code>Zbbj_test/Cards/</code> in your file browser (or run <code>!ls Zbbj_test/Cards/</code>). MadGraph automatically generates default <code>param_card.dat</code> (particle masses & widths) and copies your custom <code>run_card.dat</code> into the process cards directory.</p>

</details>

## Step 6: Les Houches Event (LHE) Structure Inspection

An LHE file stores event records in XML format.

### Key Record Columns:
- `PID`: $+5 = b$, $-5 = \bar{b}$, $23 = Z$, $21 = g$, $1..4 = u,d,c,s$.
- `Status`: `-1` (incoming parton), `+1` (outgoing final-state parton), `+2` (intermediate decayed resonance).
- `Mother1, Mother2`: Indices pointing to parent particles.
- `Px, Py, Pz, E, Mass`: Particle four-momentum components in GeV.

In [ ]:
import gzip

lhe_path = os.path.join(output_dir, "Zbbj_test/Events/run_01/unweighted_events.lhe.gz")
if not os.path.exists(lhe_path):
    lhe_path = os.path.join(output_dir, "Zbbj_LO/Events/run_01/unweighted_events.lhe.gz")

print(f"Inspecting LHE event file: {lhe_path}")
with gzip.open(lhe_path, "rt") as f:
    lines = f.readlines()

print(f"Total lines in LHE file: {len(lines)}")

event_lines = []
recording = False
for line in lines:
    if "<event>" in line:
        recording = True
    if recording:
        event_lines.append(line)
    if "</event>" in line:
        break

print("\n--- Sample First LHE Event Block ---")
print("".join(event_lines))


### Detailed Guide: Understanding the LHE Event Block

Here is the exact line-by-line breakdown of the sample `<event>` block printed above:

```text
<event>
 6      1 +1.7098000e+03 1.34422000e+02 7.54677100e-03 1.22103800e-01
        2 -1    0    0  503    0 -0.0000000000e+00 +0.0000000000e+00 +1.3056460859e+03 1.3056460859e+03 0.0000000000e+00 0.0000e+00 -1.0000e+00
       21 -1    0    0  502  503 +0.0000000000e+00 -0.0000000000e+00 -1.0425878548e+01 1.0425878548e+01 0.0000000000e+00 0.0000e+00 1.0000e+00
       23  2    1    2    0    0 -1.7288479548e+01 -9.7330351250e+01 +7.6215368438e+02 7.7391699886e+02 9.1088932974e+01 0.0000e+00 9.0000e+00
        5  1    3    3  501    0 +1.0339326251e+01 -5.6052474293e+00 +3.6958145973e+02 3.6979841244e+02 4.7000000000e+00 0.0000e+00 -1.0000e+00
       -5  1    3    3    0  501 -2.7627805799e+01 -9.1725103821e+01 +3.9257222465e+02 4.0411858642e+02 4.7000000000e+00 0.0000e+00 1.0000e+00
        2  1    1    2  502    0 +1.7288479548e+01 +9.7330351250e+01 +5.3306652301e+02 5.4215496562e+02 0.0000000000e+00 0.0000e+00 -1.0000e+00
<mgrwt>
...
</mgrwt>
</event>
```

#### 1. Header Line (`<event>` line 1)
```text
 6      1 +1.7098000e+03 1.34422000e+02 7.54677100e-03 1.22103800e-01
```
- `NUP = 6`: Number of particle entries in this event.
- `IDPRUP = 1`: Process identification index.
- `XWGTUP = +1.7098000e+03`: Monte Carlo weight $w_i$ of this event (in pb).
- `SCALUP = 134.42`: Factorization/Renormalization scale $\mu_F = \mu_R \approx 134.4\text{ GeV}$.
- `AQED = 0.007547`: Electroweak coupling $\alpha_{\text{QED}}$ at scale $\mu$.
- `AQCD = 0.122104`: Strong coupling constant $\alpha_s$ at scale $\mu$.

#### 2. Particle Entry Columns (13 Columns per Particle Line)
Each particle row contains 13 standard Les Houches fields:
`IDUP  ISTUP  MOTHUP(1) MOTHUP(2)  ICOLUP(1) ICOLUP(2)  Px  Py  Pz  E  Mass  VTIMUP  SPINUP`

- **Line 1 (`PID = 2`, `status = -1`)**: Incoming initial-state $u$ quark ($p_z = +1305.6\text{ GeV}$) from Beam 1. Color tag `503`.
- **Line 2 (`PID = 21`, `status = -1`)**: Incoming initial-state gluon ($p_z = -10.4\text{ GeV}$) from Beam 2. Color tags `502` / `503`.
- **Line 3 (`PID = 23`, `status = 2`)**: Intermediate $Z$ boson ($m_Z = 91.09\text{ GeV}$, $p_T^Z \approx 98.8\text{ GeV}$). Mother indices: `1 2` ($u g \to Z j$).
- **Line 4 (`PID = 5`, `status = 1`)**: Outgoing final-state $b$ quark ($p_T \approx 11.8\text{ GeV}$, $E \approx 369.8\text{ GeV}$, $m = 4.7\text{ GeV}$). Mother index: `3 3` (decayed from $Z$). Color tag `501`.
- **Line 5 (`PID = -5`, `status = 1`)**: Outgoing final-state $\bar{b}$ anti-quark ($p_T \approx 95.8\text{ GeV}$, $E \approx 404.1\text{ GeV}$, $m = 4.7\text{ GeV}$). Mother index: `3 3` (decayed from $Z$). Anti-color tag `501`.
- **Line 6 (`PID = 2`, `status = 1`)**: Outgoing matrix-element recoiling $u$ quark jet ($p_T \approx 98.8\text{ GeV}$, $E \approx 542.2\text{ GeV}$). Mother indices: `1 2`. Color tag `502`.

### Visualizing Feynman Diagrams (`feynman_diagrams.pdf`)

MadGraph automatically draws Feynman diagrams for all matrix elements and saves them as PostScript (`.ps`) files inside `Zbbj_test/SubProcesses/P1_<process_name>/matrix*.ps`.

#### Direct Inline Viewing:
Instead of manually downloading individual `.ps` files, run the cell below to:
1. Merge all PostScript Feynman diagrams into a **single consolidated PDF file**: `feynman_diagrams.pdf` inside your persistent output workspace directory.
2. Automatically render and display all Feynman diagrams **inline directly inside your notebook cell output**!


In [ ]:
# ── Merge all Feynman Diagrams into a Single PDF & Display Inline ───
import os
import glob
import shutil
import subprocess
from IPython.display import display, Image

# 1. Install or check Ghostscript & Poppler tools
if not shutil.which("gs") or not shutil.which("pdftoppm"):
    if 'USE_COLAB' in globals() and USE_COLAB:
        print("Installing Ghostscript & Poppler tools via apt-get (~5s)...")
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "ghostscript", "poppler-utils"], check=False)
    else:
        print("Notice: 'ghostscript' or 'pdftoppm' not found on PATH.")
        print("To render inline Feynman diagrams locally, install ghostscript & poppler-utils (e.g. 'sudo apt install ghostscript poppler-utils' or 'brew install ghostscript poppler').")

# 2. Collect all PostScript diagram files from the test run directory in output_dir
ps_files = sorted(glob.glob(os.path.join(output_dir, "Zbbj_test", "SubProcesses", "P1_*", "matrix*.ps")))

if not ps_files:
    print("Notice: No .ps Feynman diagram files found in Zbbj_test/SubProcesses/")
elif shutil.which("gs") and shutil.which("pdftoppm"):
    print(f"Found {len(ps_files)} Feynman diagram files. Merging into single PDF...")
    merged_pdf = os.path.join(output_dir, "feynman_diagrams.pdf")
    
    try:
        cmd = ["gs", "-q", "-dNOPAUSE", "-dBATCH", "-sDEVICE=pdfwrite", f"-sOutputFile={merged_pdf}"] + ps_files
        subprocess.run(cmd, check=True)
        print(f"SUCCESS: Created merged PDF -> {merged_pdf}")

        feynman_page_prefix = os.path.join(output_dir, "feynman_page")
        subprocess.run(["pdftoppm", "-png", "-r", "150", merged_pdf, feynman_page_prefix], check=True)
        png_pages = sorted(glob.glob(os.path.join(output_dir, "feynman_page-*.png")))
        
        print(f"\n--- Inline Feynman Diagram Visualizations ({len(png_pages)} pages) ---")
        for img in png_pages:
            display(Image(filename=img))

    except Exception as e:
        print(f"Notice: Could not generate inline PDF preview: {e}")


> [!IMPORTANT]
> ### Exercise 6: LHE Particle Record Identification
> In the output of the first `<event>` block above, answer the following questions:
> 1. What are the PIDs of the incoming initial-state partons ($status = -1$)?
> 2. Find the intermediate $Z$ boson ($PID = 23$). What is its status code?
> 3. Find the final-state bottom quarks ($PID = 5$ and $PID = -5$). What are their status codes?
> 4. Where is the Monte Carlo event weight $w_i$ located in the `<event>` header line?

<details>
<summary>Click to show Exercise 6 Reference Solution</summary>

<p>1. <b>Incoming Partons</b>: Quarks ($PID = 1..4, -1..-4$) or gluons ($PID = 21$) with $status = -1$.</p>
<p>2. <b>Intermediate $Z$ Boson</b>: $PID = 23$ with $status = 2$ (decayed resonance).</p>
<p>3. <b>Final-State Bottom Quarks</b>: $PID = 5$ ($b$) and $PID = -5$ ($\bar{b}$) with $status = 1$.</p>
<p>4. <b>Event Weight</b>: Located as the 3rd numerical value in the first header line of the <code>&lt;event&gt;</code> block.</p>

</details>

## Step 7: Production Run (1000 Events)

The test run passed and you have inspected the LHE event structure. Now launch the full production run for 1000 events (`Zbbj_LO`).
Set `prod_nevents` below to control how many events to generate.

The output `Zbbj_LO/Events/run_01/unweighted_events.lhe.gz` in your persistent output workspace will be parsed in **Notebook 02**.


In [ ]:
# ── Production run ────────────────────────────────────────────────────────────
# Executes the process card created in Step 4 directly; output is saved to output_dir/Zbbj_LO
import os
import shutil

prod_nevents = 1000  # <── set the number of production events here

zbbj_prod_dir = os.path.join(output_dir, "Zbbj_LO")

# 1. Execute process card from Step 4 directly with MadGraph to compile matrix elements
print(f"Executing process card ({proc_card_path})...")
!{mg5_exe} {proc_card_path}

# 2. Copy custom run card from Step 4 into Zbbj_LO/Cards/run_card.dat so ALL parameters take effect
target_prod_run_card = os.path.join(zbbj_prod_dir, "Cards", "run_card.dat")
shutil.copyfile(run_card_path, target_prod_run_card)

# 3. Create batch launch script for MadGraph
batch_prod_cmd = (
    f"launch {zbbj_prod_dir}\n"
    f"set nevents {prod_nevents}\n"
    f"set iseed 0\n"
    f"done\n"
)
batch_prod_file = os.path.join(output_dir, "cards/launch_prod.mg5")
with open(batch_prod_file, "w") as f:
    f.write(batch_prod_cmd)

# 4. Launch production event generation using MadGraph batch script
print(f"Executing {prod_nevents}-event production run (~1-2 mins)...")
!{mg5_exe} {batch_prod_file}

lhe_path = os.path.join(zbbj_prod_dir, "Events/run_01/unweighted_events.lhe.gz")
assert os.path.exists(lhe_path), f"ERROR: Missing production LHE at {lhe_path}"
file_size_kb = os.path.getsize(lhe_path) / 1024

print(f"\n==========================================")
print(f"SUCCESS: {prod_nevents}-event production run completed!")
print(f"LHE file : {lhe_path}")
print(f"File size: {file_size_kb:.2f} KB")
print(f"==========================================")


---

## Summary & Next Steps

You have successfully:
1. Configured MadGraph5_aMC@NLO matrix element generation for $pp \to Z+j, Z \to b\bar{b}$ with $p_T^Z \ge 150\text{ GeV}$.
2. Customized physics process and run cards (`zbbj_proc_card.dat` & `zbbj_run_card.dat`).
3. Completed a 10-event test run, inspected stdout integration logs, and verified the output `Zbbj_test/Cards/` directory.
4. Parsed and inspected raw Les Houches Event (LHE) XML records and visualized Feynman diagrams inline.
5. Generated the 1000-event production LHE dataset at `Zbbj_LO/Events/run_01/unweighted_events.lhe.gz` in your persistent output workspace.

**Proceed to Part 1 — Notebook 02 (`Part1_02_read_lhe_and_plot_kinematics.ipynb`)** to parse this LHE file, calculate particle 4-momenta, and plot kinematic distributions!
